# Lab 09 - Neural Network: MLP Models

This notebook applies **Lab 9 - Neural Network** using scikit-learn MLP models so the lab remains lightweight in the workspace virtual environment.


## Lab 9 concepts used

- Train a multilayer perceptron classifier for hazardous-event risk.
- Train a multilayer perceptron regressor for PM2.5 prediction.
- Use scaling, early stopping, and fixed random states.
- Compare neural-network outputs with standard metrics.

The classifier excludes `European_AQI`; the regressor excludes `European_AQI`, `Hazardous_Event`, and the PM2.5 target from inputs.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
from pathlib import Path
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != 'AML Assignment' and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
PROJECT_ROOT


In [ ]:
DATASET_FILENAME = 'global_urban_smog_pm25_hourly.csv'
matches = sorted((PROJECT_ROOT / 'Datasets').glob(f'*/{DATASET_FILENAME}'))
if not matches:
    raise FileNotFoundError(f'Could not find {DATASET_FILENAME} under {PROJECT_ROOT / "Datasets"}')
DATASET_PATH = matches[0]
data = pd.read_csv(DATASET_PATH)
data['Timestamp'] = pd.to_datetime(data['Timestamp'])
data = data.sort_values(['Timestamp', 'City']).reset_index(drop=True)
print(DATASET_PATH)
data.head()


In [ ]:
def add_time_features(df):
    out = df.copy()
    out['Timestamp'] = pd.to_datetime(out['Timestamp'])
    out = out.sort_values(['Timestamp', 'City']).reset_index(drop=True)
    out['hour'] = out['Timestamp'].dt.hour
    out['dayofweek'] = out['Timestamp'].dt.dayofweek
    out['month'] = out['Timestamp'].dt.month
    out['dayofyear'] = out['Timestamp'].dt.dayofyear
    out['is_weekend'] = out['dayofweek'].isin([5, 6]).astype(int)
    return out

def chronological_split(df, train_size=0.8):
    split_idx = int(len(df) * train_size)
    return df.iloc[:split_idx].copy(), df.iloc[split_idx:].copy()

def latest_rows(df, max_rows):
    if len(df) <= max_rows:
        return df.copy()
    return df.tail(max_rows).copy()

NUMERIC_NO_AQI = [
    'Latitude', 'Longitude', 'PM10_ug_m3', 'PM2_5_ug_m3',
    'Carbon_Monoxide_ug_m3', 'Nitrogen_Dioxide_ug_m3',
    'Ozone_ug_m3', 'Dust_ug_m3', 'UV_Index',
    'hour', 'dayofweek', 'month', 'is_weekend'
]
CATEGORICAL_FEATURES = ['City']


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_preprocess = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_preprocess = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_preprocess, NUMERIC_NO_AQI),
        ('cat', categorical_preprocess, CATEGORICAL_FEATURES),
    ],
    remainder='drop',
    verbose_feature_names_out=False,
)


In [ ]:
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, precision_recall_curve, f1_score
)


In [ ]:
from sklearn.neural_network import MLPClassifier, MLPRegressor

classification_df = latest_rows(add_time_features(data), 50000)
train_df, test_df = chronological_split(classification_df, train_size=0.8)
X_train = train_df[NUMERIC_NO_AQI + CATEGORICAL_FEATURES]
y_train = train_df['Hazardous_Event']
X_test = test_df[NUMERIC_NO_AQI + CATEGORICAL_FEATURES]
y_test = test_df['Hazardous_Event']

mlp_classifier = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('model', MLPClassifier(hidden_layer_sizes=(64, 32), activation='relu',
                            alpha=0.001, early_stopping=True, max_iter=200,
                            random_state=42))
])
mlp_classifier.fit(X_train, y_train)
mlp_pred = mlp_classifier.predict(X_test)
print(classification_report(y_test, mlp_pred, digits=3))


In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(mlp_classifier.named_steps['model'].loss_curve_)
plt.xlabel('Iteration')
plt.ylabel('Training loss')
plt.title('MLP classifier loss curve')
plt.grid(alpha=0.3)
plt.show()


In [ ]:
def add_time_features(df):
    out = df.copy()
    out['Timestamp'] = pd.to_datetime(out['Timestamp'])
    out = out.sort_values(['Timestamp', 'City']).reset_index(drop=True)
    out['hour'] = out['Timestamp'].dt.hour
    out['dayofweek'] = out['Timestamp'].dt.dayofweek
    out['month'] = out['Timestamp'].dt.month
    out['dayofyear'] = out['Timestamp'].dt.dayofyear
    out['is_weekend'] = out['dayofweek'].isin([5, 6]).astype(int)
    return out

def chronological_split(df, train_size=0.8):
    split_idx = int(len(df) * train_size)
    return df.iloc[:split_idx].copy(), df.iloc[split_idx:].copy()

def latest_rows(df, max_rows):
    if len(df) <= max_rows:
        return df.copy()
    return df.tail(max_rows).copy()

REGRESSION_NUMERIC_NO_AQI = [
    'Latitude', 'Longitude', 'PM10_ug_m3', 'Carbon_Monoxide_ug_m3',
    'Nitrogen_Dioxide_ug_m3', 'Ozone_ug_m3', 'Dust_ug_m3',
    'UV_Index', 'hour', 'dayofweek', 'month', 'is_weekend'
]
CATEGORICAL_FEATURES = ['City']


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_preprocess = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_preprocess = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_preprocess, REGRESSION_NUMERIC_NO_AQI),
        ('cat', categorical_preprocess, CATEGORICAL_FEATURES),
    ],
    remainder='drop',
    verbose_feature_names_out=False,
)


In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error


In [ ]:
regression_df = latest_rows(add_time_features(data), 50000)
train_df, test_df = chronological_split(regression_df, train_size=0.8)
X_train = train_df[REGRESSION_NUMERIC_NO_AQI + CATEGORICAL_FEATURES]
y_train = train_df['PM2_5_ug_m3']
X_test = test_df[REGRESSION_NUMERIC_NO_AQI + CATEGORICAL_FEATURES]
y_test = test_df['PM2_5_ug_m3']

mlp_regressor = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('model', MLPRegressor(hidden_layer_sizes=(64, 32), activation='relu',
                           alpha=0.001, early_stopping=True, max_iter=200,
                           random_state=42))
])
mlp_regressor.fit(X_train, y_train)
reg_pred = mlp_regressor.predict(X_test)
print('MAE:', mean_absolute_error(y_test, reg_pred))
print('RMSE:', root_mean_squared_error(y_test, reg_pred))
print('R2:', r2_score(y_test, reg_pred))


In [ ]:
plot_df = pd.DataFrame({'actual': y_test, 'predicted': reg_pred}).sample(n=min(4000, len(y_test)), random_state=42)
plt.figure(figsize=(6, 5))
sns.scatterplot(data=plot_df, x='actual', y='predicted', alpha=0.25)
plt.xlabel('Actual PM2.5')
plt.ylabel('Predicted PM2.5')
plt.title('MLP regressor: actual vs predicted')
plt.show()


## What was learned from Lab 9

Neural networks can model non-linear relationships, but they require scaling, longer training time, and careful validation. For the final assignment, their value should be judged against simpler interpretable models, not by complexity alone.
